In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_books(min_rating, max_price):
    
    base_url = "https://books.toscrape.com/"
    
    rating_map = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5
    }
    
    all_books = []
    
    # Go through all 50 pages of the website
    for page in range(1, 51):
        
        if page == 1:
            page_url = base_url
        else:
            page_url = base_url + "catalogue/page-" + str(page) + ".html"
        
        response = requests.get(page_url)
        soup = BeautifulSoup(response.content, "html.parser")
        
        books = soup.find_all("article", class_="product_pod")
        
        for book in books:
            
            # Title
            title = book.h3.a["title"]
            
            # Price
            price_text = book.find("p", class_="price_color").text
            price = float(price_text.replace("£", ""))
            
            # Rating
            rating_word = book.find("p", class_="star-rating")["class"][1]
            rating = rating_map[rating_word]
            
            # Only continue if the book matches the filters
            if rating >= min_rating and price <= max_price:
                
                # Get the link to the book detail page
                book_link = book.h3.a["href"]
                
                if book_link.startswith("catalogue/"):
                    book_url = base_url + book_link
                else:
                    book_url = base_url + "catalogue/" + book_link
                
                book_response = requests.get(book_url)
                book_soup = BeautifulSoup(book_response.content, "html.parser")
                
                # UPC from the product information table
                upc = book_soup.find("th", string="UPC").find_next("td").text
                
                # Availability
                availability = book_soup.find("th", string="Availability").find_next("td").text.strip()
                
                # Genre from breadcrumb
                genre = book_soup.select(".breadcrumb li a")[2].text.strip()
                
                # Description
                description_title = book_soup.find("div", id="product_description")
                
                if description_title:
                    description = description_title.find_next("p").text
                else:
                    description = "No description"
                
                all_books.append({
                    "UPC": upc,
                    "Title": title,
                    "Price (£)": price,
                    "Rating": rating,
                    "Genre": genre,
                    "Availability": availability,
                    "Description": description
                })
    
    df = pd.DataFrame(all_books)
    
    return df

In [ ]:
df = scrape_books(4, 2 0)
df

,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,ce6396b0f23f6ecc,Set Me Free,17.46,5,Young Adult,In stock (19 available),Aaron Ledbetter’s future had been planned out ...
1,6258a1f6a6dcfe50,The Four Agreements: A Practical Guide to Pers...,17.66,5,Spirituality,In stock (18 available),"In The Four Agreements, don Miguel Ruiz reveal..."
2,6be3beb0793a53e7,Sophie's World,15.94,5,Philosophy,In stock (18 available),A page-turning novel that is also an explorati...
3,657fe5ead67a7767,Untitled Collection: Sabbath Poems 2014,14.27,4,Poetry,In stock (16 available),"More than thirty-five years ago, when the weat..."
4,51653ef291ab7ddc,This One Summer,19.49,4,Sequential Art,In stock (16 available),"Every summer, Rose goes with her mom and dad t..."
...,...,...,...,...,...,...,...
70,9c96cd1329fbd82d,The Zombie Room,19.69,5,Default,In stock (1 available),An unlikely bond is forged between three men f...
71,b78deb463531d078,The Silent Wife,12.34,5,Fiction,In stock (1 available),A chilling psychological thriller about a marr...
72,4280ac3eab57aa5d,The Girl You Lost,12.29,5,Mystery,In stock (1 available),Eighteen years ago your baby daughter was snat...
73,29fc016c459aeb14,The Edge of Reason (Bridget Jones #2),19.18,4,Womens Fiction,In stock (1 available),Monday 27 January“7:15 a.m. Hurrah! The wilder...
